In [1]:
import pandas as pd
import os
from os.path import join as pjoin
import os
import cv2
import numpy as np
from tqdm import tqdm


In [12]:
path_face = '/home/lzx/data/WF_VC_liuzhaoxi/25.8.7_A9/natural-movie/raw/20250807-062427-event'

experiment = os.path.basename(path_face)[:15]
mouse = os.path.normpath(path_face).split(os.sep)[5].split('_')[1]
path_out = pjoin(path_face, '../../process', experiment + '-natural-movie--/face')
os.makedirs(path_out, exist_ok=True)
print(experiment,mouse)
trialfile = pd.read_csv(pjoin(path_out, f'../../{experiment}-wfield/trials.csv'), header=None).values.astype(int)
movie_folder = pjoin(path_face, '../natural_movies')
movie_list = pd.read_csv(pjoin(movie_folder, 'movie_list.txt'), header=None).values
n_movie = movie_list.size
movie_name_list = []
for imovie in range(n_movie):
    movie_name_list.append(str(movie_list[imovie])[11:-6])

n_frame = 150  # 帧
pre_length = 20
after_length = 30


20250807-062427 A9


In [13]:
# 创建14个文件夹
for i, movie_name in enumerate(movie_name_list):
    folder_name = f'{i}_{movie_name}'
    folder_path = os.path.join(path_out, folder_name)
    os.makedirs(folder_path, exist_ok=True)


In [14]:
# 处理每个trial
for movie_idx in range(n_movie):  # 14个movie
    for repeat_idx in range(5):  # 每个movie重复5次
        trial_idx = movie_idx * 5 + repeat_idx  # 计算当前trial的索引(0-69)
        start_frame = trialfile[trial_idx, 1]  # 获取开始帧
        
        # 计算起始和结束帧
        frame_start = start_frame - pre_length
        frame_end = start_frame + n_frame + after_length
        
        # 准备视频写入器
        folder_name = f'{movie_idx}_{movie_name_list[movie_idx]}'
        output_path = os.path.join(path_out, folder_name, f'{repeat_idx}.mp4')
        
        # 获取第一帧来确定视频尺寸
        first_frame_path = os.path.join(path_face, f"{frame_start}.tif")
        first_frame = cv2.imread(first_frame_path)
        if first_frame is None:
            print(f"无法读取第一帧: {first_frame_path}")
            continue
        
        height, width, _ = first_frame.shape
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, 30.0, (width, height))
        
        # 写入每一帧
        for frame_num in range(frame_start, frame_end + 1):
            frame_path = os.path.join(path_face, f"{frame_num}.tif")
            frame = cv2.imread(frame_path)
            if frame is None:
                print(f"无法读取帧: {frame_path}")
                break
            out.write(frame)
        
        out.release()
        print(f"已完成: {output_path}")

print("所有视频处理完成!")

已完成: /home/lzx/data/WF_VC_liuzhaoxi/25.8.7_A9/natural-movie/raw/20250807-062427-event/../../process/20250807-062427-natural-movie--/face/0_forest/0.mp4
已完成: /home/lzx/data/WF_VC_liuzhaoxi/25.8.7_A9/natural-movie/raw/20250807-062427-event/../../process/20250807-062427-natural-movie--/face/0_forest/1.mp4
已完成: /home/lzx/data/WF_VC_liuzhaoxi/25.8.7_A9/natural-movie/raw/20250807-062427-event/../../process/20250807-062427-natural-movie--/face/0_forest/2.mp4
已完成: /home/lzx/data/WF_VC_liuzhaoxi/25.8.7_A9/natural-movie/raw/20250807-062427-event/../../process/20250807-062427-natural-movie--/face/0_forest/3.mp4
已完成: /home/lzx/data/WF_VC_liuzhaoxi/25.8.7_A9/natural-movie/raw/20250807-062427-event/../../process/20250807-062427-natural-movie--/face/0_forest/4.mp4
已完成: /home/lzx/data/WF_VC_liuzhaoxi/25.8.7_A9/natural-movie/raw/20250807-062427-event/../../process/20250807-062427-natural-movie--/face/1_Heron/0.mp4
已完成: /home/lzx/data/WF_VC_liuzhaoxi/25.8.7_A9/natural-movie/raw/20250807-062427-event/../

In [15]:
# 处理每个trial

from tifffile import imread, imwrite

for movie_idx in tqdm(range(n_movie), desc="Processing movies"):
    for repeat_idx in range(5):
        trial_idx = movie_idx * 5 + repeat_idx
        start_frame = trialfile[trial_idx, 1]
        
        # 计算帧范围
        frame_range = range(
            start_frame - pre_length,
            start_frame + n_frame + after_length
        )
        
        # 读取所有帧到内存
        frames = []
        for frame_num in frame_range:
            frame_path = os.path.join(path_face, f"{frame_num}.tif")
            frame = imread(frame_path)
            frames.append(frame)
            
        # 转换为3D numpy数组 (frames, height, width)
        stack = np.stack(frames)
        
        # 输出路径
        output_dir = os.path.join(path_out, f'{movie_idx}_{movie_name_list[movie_idx]}')
        output_path = os.path.join(output_dir, f'{repeat_idx}.tif')
        
        # 写入TIFF stack (可添加压缩参数)
        imwrite(
            output_path,
            stack,
            metadata={'description': f'Movie {movie_idx}, Trial {repeat_idx}'}
        )

print("所有TIFF stack处理完成!")

Processing movies: 100%|██████████| 14/14 [00:19<00:00,  1.42s/it]

所有TIFF stack处理完成!
